In [ ]:
from huggingface_hub import login

login(token="YOUR_HF_TOKEN_HERE")

In [ ]:
import os
import shutil
from huggingface_hub import snapshot_download

# 1. Completely erase the corrupted hidden cache
cache_path = os.path.expanduser("~/.cache/huggingface/hub/models--smtiitm--Fastspeech2_HS")
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("✅ Cleared corrupted Hugging Face cache.")

# 2. Download sequentially to bypass Colab's network throttling
print("⏳ Downloading files sequentially (this is safer and won't drop)...")
snapshot_download(
    repo_id="smtiitm/Fastspeech2_HS",
    allow_patterns=[
        "assamese/**",
        "bengali/**",
        "vocoder/**",
        "*.py",
        "*.json",
        "*.txt"
    ],
    local_dir="/content/Fastspeech2_HS",
    local_dir_use_symlinks=False,
    force_download = True,
    max_workers=1  # <-- This prevents Colab from killing the connection
)

print("🎉 Setup Complete! The files downloaded successfully.")

✅ Cleared corrupted Hugging Face cache.
⏳ Downloading files sequentially (this is safer and won't drop)...


Fetching 116 files:   0%|          | 0/116 [00:00<?, ?it/s]

OSError: Consistency check failed: file should be of size 55829161 but has size 55786428 (generator).
This is usually due to network issues while downloading the file. Please retry with `force_download=True`.

In [4]:
import os
import subprocess

# 1. Install huggingface_hub to read the repository map
os.system("pip install -q huggingface_hub")
from huggingface_hub import HfApi

repo_id = "smtiitm/Fastspeech2_HS"
base_dir = "/content/Fastspeech2_HS"

print("🧹 Cleaning up previous attempts...")
os.system(f"rm -rf {base_dir}")
os.makedirs(base_dir, exist_ok=True)

print("📋 Reading repository structure from Hugging Face...")
api = HfApi()
all_files = api.list_repo_files(repo_id)

# FIXED FILTER: Grab the specific weights, PLUS every single Python/Config file everywhere.
needed_files = [
    f for f in all_files
    if f.startswith(("assamese/", "bengali/", "vocoder/"))
    or f.endswith((".py", ".json", ".txt"))
]

print(f"🚀 Starting bulletproof download of {len(needed_files)} files...")

for i, file_path in enumerate(needed_files, 1):
    dest_path = os.path.join(base_dir, file_path)
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

    url = f"https://huggingface.co/{repo_id}/resolve/main/{file_path}"
    print(f"[{i}/{len(needed_files)}] Downloading: {file_path}")

    # Robust WGET Loop with native Resume (-c)
    max_retries = 10
    for attempt in range(max_retries):
        cmd = f"wget -c -q --show-progress '{url}' -O '{dest_path}'"
        result = subprocess.run(cmd, shell=True)

        if result.returncode == 0:
            break
        else:
            print(f"   ⚠️ Drop detected. Resuming bytes (Attempt {attempt+1}/{max_retries})...")

    if result.returncode != 0:
        print(f"❌ Failed to download {file_path}")

print("\n🎉 SETUP COMPLETE! All source code and weights are safely downloaded.")

🧹 Cleaning up previous attempts...
📋 Reading repository structure from Hugging Face...
🚀 Starting bulletproof download of 116 files...
[1/116] Downloading: Unified_parser/.vscode/tasks.json
[2/116] Downloading: Unified_parser/extract_words.py
[3/116] Downloading: Unified_parser/get_phone_mapped_python.py
[4/116] Downloading: Unified_parser/globals.py
[5/116] Downloading: Unified_parser/helpers.py
[6/116] Downloading: Unified_parser/ply/__init__.py
[7/116] Downloading: Unified_parser/ply/lex.py
[8/116] Downloading: Unified_parser/ply/yacc.py
[9/116] Downloading: Unified_parser/punjabi/extract_punjabi.py
[10/116] Downloading: Unified_parser/punjabi/punjabi_results.txt
[11/116] Downloading: Unified_parser/punjabi/punjabi_words.txt
[12/116] Downloading: Unified_parser/punjabi/runner_punjabi.py
[13/116] Downloading: Unified_parser/pypi_package/build/lib/indic_unified_parser/__init__.py
[14/116] Downloading: Unified_parser/pypi_package/build/lib/indic_unified_parser/globals.py
[15/116] Downl

In [2]:
!pip install -q espnet typeguard==2.13.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 16.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 21.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
!pip install -q indic-num2words

In [10]:
!pip install -q indic_unified_parser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.6 MB/s eta 0:00:00


In [11]:
import json
import os
import subprocess
import shutil
import torch
import librosa
from jiwer import wer, cer
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from google.colab import files

# ==========================================
# 1. CONFIGURATION
# ==========================================
# CHANGE THIS to "assamese" or "bengali"
TARGET_LANGUAGE = "assamese"

JSON_PATH = f"/content/{TARGET_LANGUAGE}_evaluation_set.json"
OUTPUT_DIR = f"/content/output_audio_{TARGET_LANGUAGE}_fs2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Map text keys based on your uploaded JSON formats
if TARGET_LANGUAGE == "assamese":
    text_key = "assamese_sentence"
elif TARGET_LANGUAGE == "bengali":
    text_key = "bengali_sentence"

# ==========================================
# 2. GENERATION LOOP (FastSpeech2)
# ==========================================
print(f"\n🚀 Starting FastSpeech2 TTS Generation for {TARGET_LANGUAGE.upper()}...")
print("-" * 60)

try:
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        dataset = json.load(f)
except FileNotFoundError:
    print(f"❌ Error: Could not find '{JSON_PATH}'. Please ensure it is uploaded.")
    dataset = []

# Handle list-based JSON structures
eval_items = [{"id": item.get("id", item.get("ID")), "text": item.get(text_key, "")} for item in dataset]

for item in eval_items:
    item_id = str(item["id"])
    text = item["text"]

    if not text:
        continue

    filename = os.path.join(OUTPUT_DIR, f"{item_id}.wav")
    print(f"Processing ID: {item_id}")

    # We call the IITM inference.py script via command line
    process = subprocess.Popen(
        [
            "python", "inference.py",
            "--sample_text", text,
            "--language", TARGET_LANGUAGE,
            "--gender", "female", # The models include 'male' and 'female'
            "--alpha", "1.0",     # 1.0 is normal speed
            "--output_file", filename
        ],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        cwd="/content/Fastspeech2_HS" # CRITICAL: Must be run inside the cloned repo
    )

    stdout, stderr = process.communicate()

    if process.returncode == 0:
        print(f"✅ Success: {item_id}.wav")
    else:
        print(f"❌ FAILED: {stderr.decode('utf-8')}")

# ==========================================
# 3. WHISPER EVALUATION
# ==========================================
print(f"\n🚀 Starting Whisper Evaluation for {TARGET_LANGUAGE.upper()}...")
print("-" * 60)

device = "cuda" if torch.cuda.is_available() else "cpu"
asr_processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
asr_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium").to(device)
forced_decoder_ids = asr_processor.get_decoder_prompt_ids(language=TARGET_LANGUAGE, task="transcribe")

results = []

for item in eval_items:
    item_id = str(item["id"])
    ref_text = item["text"]
    file_path = os.path.join(OUTPUT_DIR, f"{item_id}.wav")

    if not os.path.exists(file_path):
        continue

    # Transcribe Audio
    speech_array, sampling_rate = librosa.load(file_path, sr=16000)
    input_features = asr_processor(speech_array, sampling_rate=16000, return_tensors="pt").input_features.to(device)

    with torch.no_grad():
        predicted_ids = asr_model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

    hyp_text = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Normalize Text
    norm_ref = ref_text.replace("।", "").replace(",", "").replace("?", "").replace("!", "").strip()
    norm_hyp = hyp_text.replace("।", "").replace(",", "").replace("?", "").replace("!", "").strip()

    try:
        item_wer = wer(norm_ref, norm_hyp)
        item_cer = cer(norm_ref, norm_hyp)
    except ValueError:
        item_wer, item_cer = 1.0, 1.0

    print(f"ID {item_id}")
    print(f"  Ref: {norm_ref}")
    print(f"  Hyp: {norm_hyp}")
    print(f"  --> WER: {item_wer:.3f} | CER: {item_cer:.3f}\n")
    results.append({"wer": item_wer, "cer": item_cer})

# Calculate final averages
if results:
    avg_wer = sum(r["wer"] for r in results) / len(results)
    avg_cer = sum(r["cer"] for r in results) / len(results)
    print("-" * 60)
    print(f"🎯 FASTSPEECH2 {TARGET_LANGUAGE.upper()} AVERAGES: WER = {avg_wer:.3f} | CER = {avg_cer:.3f}")
    print("-" * 60)

# ==========================================
# 4. ZIP AND DOWNLOAD
# ==========================================
zip_filename = f"{TARGET_LANGUAGE}_fastspeech2_audio"
zip_path = f"/content/{zip_filename}"

if len(os.listdir(OUTPUT_DIR)) > 0:
    print(f"\n📦 Zipping the '{OUTPUT_DIR}' directory...")
    shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
    print("⬇️ Triggering download to your local machine...")
    files.download(f"{zip_path}.zip")
else:
    print("⚠️ No audio files were generated to zip.")


🚀 Starting FastSpeech2 TTS Generation for ASSAMESE...
------------------------------------------------------------
Processing ID: 1
✅ Success: 1.wav
Processing ID: 2
✅ Success: 2.wav
Processing ID: 3
✅ Success: 3.wav
Processing ID: 4
✅ Success: 4.wav
Processing ID: 5
✅ Success: 5.wav
Processing ID: 6
✅ Success: 6.wav
Processing ID: 7
✅ Success: 7.wav
Processing ID: 8
❌ FAILED: /usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Traceback (most recent call last):
  File "/content/Fastspeech2_HS/inference.py", line 119, in <module>
    preprocessed_text, phrases = preprocessor.preprocess(args.sample_text, args.language, args.gender, phone_dictionary)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Fastspeech2_HS/text_prepro

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

KeyboardInterrupt: Keyboard interrupt (SIGINT)

In [12]:
import json
import os
import subprocess
import shutil
import torch
import librosa
from jiwer import wer, cer
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from google.colab import files

# ==========================================
# 1. CONFIGURATION
# ==========================================
# CHANGE THIS to "assamese" or "bengali"
TARGET_LANGUAGE = "bengali"

JSON_PATH = f"/content/{TARGET_LANGUAGE}_evaluation_set.json"
OUTPUT_DIR = f"/content/output_audio_{TARGET_LANGUAGE}_fs2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Map text keys based on your uploaded JSON formats
if TARGET_LANGUAGE == "assamese":
    text_key = "assamese_sentence"
elif TARGET_LANGUAGE == "bengali":
    text_key = "bengali_sentence"

# ==========================================
# 2. GENERATION LOOP (FastSpeech2)
# ==========================================
print(f"\n🚀 Starting FastSpeech2 TTS Generation for {TARGET_LANGUAGE.upper()}...")
print("-" * 60)

try:
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        dataset = json.load(f)
except FileNotFoundError:
    print(f"❌ Error: Could not find '{JSON_PATH}'. Please ensure it is uploaded.")
    dataset = []

# Handle list-based JSON structures
eval_items = [{"id": item.get("id", item.get("ID")), "text": item.get(text_key, "")} for item in dataset]

for item in eval_items:
    item_id = str(item["id"])
    text = item["text"]

    if not text:
        continue

    filename = os.path.join(OUTPUT_DIR, f"{item_id}.wav")
    print(f"Processing ID: {item_id}")

    # We call the IITM inference.py script via command line
    process = subprocess.Popen(
        [
            "python", "inference.py",
            "--sample_text", text,
            "--language", TARGET_LANGUAGE,
            "--gender", "female", # The models include 'male' and 'female'
            "--alpha", "1.0",     # 1.0 is normal speed
            "--output_file", filename
        ],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        cwd="/content/Fastspeech2_HS" # CRITICAL: Must be run inside the cloned repo
    )

    stdout, stderr = process.communicate()

    if process.returncode == 0:
        print(f"✅ Success: {item_id}.wav")
    else:
        print(f"❌ FAILED: {stderr.decode('utf-8')}")

# ==========================================
# 3. WHISPER EVALUATION
# ==========================================
print(f"\n🚀 Starting Whisper Evaluation for {TARGET_LANGUAGE.upper()}...")
print("-" * 60)

device = "cuda" if torch.cuda.is_available() else "cpu"
asr_processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
asr_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium").to(device)
forced_decoder_ids = asr_processor.get_decoder_prompt_ids(language=TARGET_LANGUAGE, task="transcribe")

results = []

for item in eval_items:
    item_id = str(item["id"])
    ref_text = item["text"]
    file_path = os.path.join(OUTPUT_DIR, f"{item_id}.wav")

    if not os.path.exists(file_path):
        continue

    # Transcribe Audio
    speech_array, sampling_rate = librosa.load(file_path, sr=16000)
    input_features = asr_processor(speech_array, sampling_rate=16000, return_tensors="pt").input_features.to(device)

    with torch.no_grad():
        predicted_ids = asr_model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

    hyp_text = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Normalize Text
    norm_ref = ref_text.replace("।", "").replace(",", "").replace("?", "").replace("!", "").strip()
    norm_hyp = hyp_text.replace("।", "").replace(",", "").replace("?", "").replace("!", "").strip()

    try:
        item_wer = wer(norm_ref, norm_hyp)
        item_cer = cer(norm_ref, norm_hyp)
    except ValueError:
        item_wer, item_cer = 1.0, 1.0

    print(f"ID {item_id}")
    print(f"  Ref: {norm_ref}")
    print(f"  Hyp: {norm_hyp}")
    print(f"  --> WER: {item_wer:.3f} | CER: {item_cer:.3f}\n")
    results.append({"wer": item_wer, "cer": item_cer})

# Calculate final averages
if results:
    avg_wer = sum(r["wer"] for r in results) / len(results)
    avg_cer = sum(r["cer"] for r in results) / len(results)
    print("-" * 60)
    print(f"🎯 FASTSPEECH2 {TARGET_LANGUAGE.upper()} AVERAGES: WER = {avg_wer:.3f} | CER = {avg_cer:.3f}")
    print("-" * 60)

# ==========================================
# 4. ZIP AND DOWNLOAD
# ==========================================
zip_filename = f"{TARGET_LANGUAGE}_fastspeech2_audio"
zip_path = f"/content/{zip_filename}"

if len(os.listdir(OUTPUT_DIR)) > 0:
    print(f"\n📦 Zipping the '{OUTPUT_DIR}' directory...")
    shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
    print("⬇️ Triggering download to your local machine...")
    files.download(f"{zip_path}.zip")
else:
    print("⚠️ No audio files were generated to zip.")


🚀 Starting FastSpeech2 TTS Generation for BENGALI...
------------------------------------------------------------
Processing ID: 1
✅ Success: 1.wav
Processing ID: 2
✅ Success: 2.wav
Processing ID: 3
✅ Success: 3.wav
Processing ID: 4
✅ Success: 4.wav
Processing ID: 5
✅ Success: 5.wav
Processing ID: 6
✅ Success: 6.wav
Processing ID: 7
✅ Success: 7.wav
Processing ID: 8
❌ FAILED: /usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Traceback (most recent call last):
  File "/content/Fastspeech2_HS/inference.py", line 119, in <module>
    preprocessed_text, phrases = preprocessor.preprocess(args.sample_text, args.language, args.gender, phone_dictionary)
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Fastspeech2_HS/text_preproc

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressToken

ID 1
  Ref: অন্ধকার ঘরে কাঁপা হাতে সে পুরোনো পুঁথি আর রথের ভাঙা চাকা খুঁজছে
  Hyp: अन्धुकार गौरे कापाहाते से पुरोनो पुढियार रथेर भागा जका खुज्जे
  --> WER: 1.000 | CER: 0.889

ID 2
  Ref: ফাল্গুনের মেলায় ভণ্ড সাধুর কথায় ঘণ্টা বাজতেই এক অদ্ভুত কম্পন তৈরি হলো
  Hyp: फाल्कुनेर मेलाय भान्ड्षादुर कथाय गहांटा बाज़ती एकद्भूत कंपुन तुईरी हुलो
  --> WER: 1.000 | CER: 0.928

ID 3
  Ref: শহরের এই উচ্চ অট্টালিকার ছাদে দাঁড়ালে উদ্দাম বাতাসের শব্দে অন্য সত্তার খোঁজ মেলে
  Hyp: សវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវវ
  --> WER: 1.000 | CER: 1.850

ID 4
  Ref: বর্ষার শেষে মেঘলা আকাশে হঠাৎ এক ঝাঁক সাদা বক আর সবুজ ঘাসের ওপর ব্যাঙের ডাক শোনা গেল
  Hyp: बर्षा शेषे मेगला अकाशे हथातेक जाक्ष अदाबोकार सबूझ घाशे रुपर बैंगे डाक्षो ना गयलो
  --> WER: 1.000 | CER: 0.904

ID 5
  Ref: সে রেগে গিয়ে বলল "আমার ছাতা আর ফলের ঝুড়িটা কোথায় জেনেশুনে লুকিয়ে রেখেছ"
  Hyp: शेरे गी गी भूल्लो और अमार छाता और फलेर जुटव

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import os
import json
import torch
import librosa
import shutil
import pandas as pd
from jiwer import wer, cer
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from google.colab import files

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Change this to "assamese" or "bengali" based on what you are evaluating
TARGET_LANGUAGE = "assamese"

JSON_PATH = f"/content/{TARGET_LANGUAGE}_evaluation_set.json"
OUTPUT_DIR = f"/content/output_audio_{TARGET_LANGUAGE}_fs2"

# Determine the correct JSON key based on the language
if TARGET_LANGUAGE == "assamese":
    text_key = "assamese_sentence"
elif TARGET_LANGUAGE == "bengali":
    text_key = "bengali_sentence"
else:
    text_key = "text"

# ==========================================
# 2. LOAD ASR MODEL & DATA
# ==========================================
print(f"⏳ Loading Whisper Medium ASR model for {TARGET_LANGUAGE.upper()}...")
device = "cuda" if torch.cuda.is_available() else "cpu"

asr_processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
asr_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium").to(device)
forced_decoder_ids = asr_processor.get_decoder_prompt_ids(language=TARGET_LANGUAGE, task="transcribe")

# Load Ground Truth
try:
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    # Handle list format
    ground_truth = {str(item.get("id", item.get("ID"))): item.get(text_key, "") for item in dataset}
    print(f"✅ Loaded {len(ground_truth)} reference sentences.")
except Exception as e:
    print(f"❌ Error loading JSON: {e}")
    ground_truth = {}

# ==========================================
# 3. UNATTENDED EVALUATION LOOP
# ==========================================
print("\n🚀 Starting unattended evaluation. Go enjoy your lunch!")
print("-" * 60)

results_data = []

# Process all WAV files in the directory
if os.path.exists(OUTPUT_DIR):
    for filename in os.listdir(OUTPUT_DIR):
        if not filename.endswith(".wav"):
            continue

        file_path = os.path.join(OUTPUT_DIR, filename)
        item_id = filename.replace(".wav", "")
        ref_text = ground_truth.get(item_id, "")

        if not ref_text:
            continue

        # Transcribe
        speech_array, sampling_rate = librosa.load(file_path, sr=16000)
        input_features = asr_processor(speech_array, sampling_rate=16000, return_tensors="pt").input_features.to(device)

        with torch.no_grad():
            predicted_ids = asr_model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

        hyp_text = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        # Normalize for accurate metrics
        norm_ref = ref_text.replace("।", "").replace(",", "").replace("?", "").replace("!", "").strip()
        norm_hyp = hyp_text.replace("।", "").replace(",", "").replace("?", "").replace("!", "").strip()

        try:
            item_wer = wer(norm_ref, norm_hyp)
            item_cer = cer(norm_ref, norm_hyp)
        except ValueError:
            item_wer, item_cer = 1.0, 1.0

        results_data.append({
            "ID": item_id,
            "Reference": norm_ref,
            "Prediction": norm_hyp,
            "WER": round(item_wer, 4),
            "CER": round(item_cer, 4)
        })

        # Print progress to the console just in case
        print(f"Processed {filename} -> WER: {item_wer:.3f} | CER: {item_cer:.3f}")

# ==========================================
# 4. SAVE METRICS, ZIP, AND AUTO-DOWNLOAD
# ==========================================
if results_data:
    # Save a CSV report INSIDE the audio folder so it gets zipped together
    csv_path = os.path.join(OUTPUT_DIR, f"{TARGET_LANGUAGE}_metrics_report.csv")
    df = pd.DataFrame(results_data)
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')

    avg_wer = df["WER"].mean()
    avg_cer = df["CER"].mean()
    print("-" * 60)
    print(f"🎯 FINAL AVERAGES: WER = {avg_wer:.3f} | CER = {avg_cer:.3f}")
    print(f"📄 Report saved to {csv_path}")
    print("-" * 60)

    # Zip the entire folder
    zip_path = f"/content/{TARGET_LANGUAGE}_evaluation_complete"
    print(f"📦 Zipping all audio files and metrics into {zip_path}.zip...")
    shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)

    # Trigger the browser download automatically
    print("⬇️ Triggering automated download...")
    files.download(f"{zip_path}.zip")
    print("✅ Process complete!")
else:
    print("⚠️ No audio files found to evaluate.")

⏳ Loading Whisper Medium ASR model for ASSAMESE...


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

✅ Loaded 20 reference sentences.

🚀 Starting unattended evaluation. Go enjoy your lunch!
------------------------------------------------------------
Processed 6.wav -> WER: 1.000 | CER: 1.013
Processed 9.wav -> WER: 9.933 | CER: 3.388
Processed 7.wav -> WER: 1.083 | CER: 1.072
Processed 20.wav -> WER: 1.000 | CER: 0.887
Processed 4.wav -> WER: 1.067 | CER: 0.867
Processed 11.wav -> WER: 1.071 | CER: 0.917
Processed 16.wav -> WER: 1.000 | CER: 0.871
Processed 17.wav -> WER: 11.385 | CER: 4.101
Processed 1.wav -> WER: 1.000 | CER: 0.862
Processed 12.wav -> WER: 1.385 | CER: 1.125
Processed 13.wav -> WER: 1.000 | CER: 0.899
Processed 19.wav -> WER: 1.000 | CER: 1.029
Processed 2.wav -> WER: 1.000 | CER: 1.040
Processed 18.wav -> WER: 1.333 | CER: 0.959
Processed 3.wav -> WER: 1.000 | CER: 0.966
Processed 5.wav -> WER: 1.308 | CER: 0.959
Processed 15.wav -> WER: 1.067 | CER: 0.869
Processed 10.wav -> WER: 1.118 | CER: 0.951
Processed 14.wav -> WER: 1.083 | CER: 0.900
---------------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Process complete!


In [14]:
import os
import json
import torch
import librosa
import shutil
import pandas as pd
from jiwer import wer, cer
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from google.colab import files

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Change this to "assamese" or "bengali" based on what you are evaluating
TARGET_LANGUAGE = "bengali"

JSON_PATH = f"/content/{TARGET_LANGUAGE}_evaluation_set.json"
OUTPUT_DIR = f"/content/output_audio_{TARGET_LANGUAGE}_fs2"

# Determine the correct JSON key based on the language
if TARGET_LANGUAGE == "assamese":
    text_key = "assamese_sentence"
elif TARGET_LANGUAGE == "bengali":
    text_key = "bengali_sentence"
else:
    text_key = "text"

# ==========================================
# 2. LOAD ASR MODEL & DATA
# ==========================================
print(f"⏳ Loading Whisper Medium ASR model for {TARGET_LANGUAGE.upper()}...")
device = "cuda" if torch.cuda.is_available() else "cpu"

asr_processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
asr_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium").to(device)
forced_decoder_ids = asr_processor.get_decoder_prompt_ids(language=TARGET_LANGUAGE, task="transcribe")

# Load Ground Truth
try:
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        dataset = json.load(f)
    # Handle list format
    ground_truth = {str(item.get("id", item.get("ID"))): item.get(text_key, "") for item in dataset}
    print(f"✅ Loaded {len(ground_truth)} reference sentences.")
except Exception as e:
    print(f"❌ Error loading JSON: {e}")
    ground_truth = {}

# ==========================================
# 3. UNATTENDED EVALUATION LOOP
# ==========================================
print("\n🚀 Starting unattended evaluation. Go enjoy your lunch!")
print("-" * 60)

results_data = []

# Process all WAV files in the directory
if os.path.exists(OUTPUT_DIR):
    for filename in os.listdir(OUTPUT_DIR):
        if not filename.endswith(".wav"):
            continue

        file_path = os.path.join(OUTPUT_DIR, filename)
        item_id = filename.replace(".wav", "")
        ref_text = ground_truth.get(item_id, "")

        if not ref_text:
            continue

        # Transcribe
        speech_array, sampling_rate = librosa.load(file_path, sr=16000)
        input_features = asr_processor(speech_array, sampling_rate=16000, return_tensors="pt").input_features.to(device)

        with torch.no_grad():
            predicted_ids = asr_model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

        hyp_text = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        # Normalize for accurate metrics
        norm_ref = ref_text.replace("।", "").replace(",", "").replace("?", "").replace("!", "").strip()
        norm_hyp = hyp_text.replace("।", "").replace(",", "").replace("?", "").replace("!", "").strip()

        try:
            item_wer = wer(norm_ref, norm_hyp)
            item_cer = cer(norm_ref, norm_hyp)
        except ValueError:
            item_wer, item_cer = 1.0, 1.0

        results_data.append({
            "ID": item_id,
            "Reference": norm_ref,
            "Prediction": norm_hyp,
            "WER": round(item_wer, 4),
            "CER": round(item_cer, 4)
        })

        # Print progress to the console just in case
        print(f"Processed {filename} -> WER: {item_wer:.3f} | CER: {item_cer:.3f}")

# ==========================================
# 4. SAVE METRICS, ZIP, AND AUTO-DOWNLOAD
# ==========================================
if results_data:
    # Save a CSV report INSIDE the audio folder so it gets zipped together
    csv_path = os.path.join(OUTPUT_DIR, f"{TARGET_LANGUAGE}_metrics_report.csv")
    df = pd.DataFrame(results_data)
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')

    avg_wer = df["WER"].mean()
    avg_cer = df["CER"].mean()
    print("-" * 60)
    print(f"🎯 FINAL AVERAGES: WER = {avg_wer:.3f} | CER = {avg_cer:.3f}")
    print(f"📄 Report saved to {csv_path}")
    print("-" * 60)

    # Zip the entire folder
    zip_path = f"/content/{TARGET_LANGUAGE}_evaluation_complete"
    print(f"📦 Zipping all audio files and metrics into {zip_path}.zip...")
    shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)

    # Trigger the browser download automatically
    print("⬇️ Triggering automated download...")
    files.download(f"{zip_path}.zip")
    print("✅ Process complete!")
else:
    print("⚠️ No audio files found to evaluate.")

⏳ Loading Whisper Medium ASR model for BENGALI...


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

✅ Loaded 20 reference sentences.

🚀 Starting unattended evaluation. Go enjoy your lunch!
------------------------------------------------------------
Processed 6.wav -> WER: 12.333 | CER: 3.595
Processed 9.wav -> WER: 1.000 | CER: 2.000
Processed 7.wav -> WER: 1.000 | CER: 1.682
Processed 20.wav -> WER: 1.000 | CER: 0.930
Processed 4.wav -> WER: 1.000 | CER: 0.904
Processed 11.wav -> WER: 1.000 | CER: 0.937
Processed 16.wav -> WER: 1.000 | CER: 0.930
Processed 17.wav -> WER: 1.091 | CER: 0.879
Processed 1.wav -> WER: 1.000 | CER: 0.889
Processed 12.wav -> WER: 1.077 | CER: 0.967
Processed 13.wav -> WER: 1.000 | CER: 0.879
Processed 19.wav -> WER: 1.000 | CER: 2.000
Processed 2.wav -> WER: 1.000 | CER: 0.928
Processed 18.wav -> WER: 1.000 | CER: 1.805
Processed 3.wav -> WER: 1.000 | CER: 1.850
Processed 5.wav -> WER: 1.385 | CER: 0.959
Processed 15.wav -> WER: 1.000 | CER: 0.976
Processed 10.wav -> WER: 1.000 | CER: 0.865
Processed 14.wav -> WER: 1.429 | CER: 0.886
---------------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Process complete!
